# InternVL2-26B 적용 버전

베이스라인(Qwen2.5-VL-3B)에서 InternVL2-26B로 교체한 버전입니다.

**변경 사항 요약:**
- 모델: `Qwen2.5-VL-3B` → `OpenGVLab/InternVL2-26B`
- 모델 클래스: `AutoModelForImageTextToText` → `AutoModel`
- Processor: `AutoProcessor` → `AutoTokenizer` + 직접 이미지 전처리
- 이미지 해상도: 동적 448×448 타일 분할 방식
- LoRA rank: 16 유지
- 학습률: 2e-5 유지
- Epoch: 3 유지
- compute_dtype: bfloat16 (A100 최적화)

A100 GPU (40GB) 환경 / Colab Pro에서 실행하세요.

# 환경 준비

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
!pip -q install "transformers>=4.37.2,<5" "accelerate>=0.26.0"
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade
!pip -q install einops timm  # InternVL2 필수 의존성

In [ ]:
import torch
import transformers
import pandas
import PIL

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Pandas version:", pandas.__version__)
print("Pillow version:", PIL.__version__)
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("GPU not available")

In [ ]:
# 압축 해제
!unzip "/content/2026_15_2_ai_DataSet.zip" -d "/content/"

In [ ]:
!ls /content/
!find /content -name "train.csv"
!find /content -name "test.csv"

# 라이브러리, 데이터, 설정

In [ ]:
import os, math, random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any, List
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_ID       = "OpenGVLab/InternVL2-26B"
IMG_SIZE       = 448   # InternVL2 기본 타일 크기
MAX_TILES      = 6     # 최대 타일 수 (A100 40GB 기준 안전값)
MAX_NEW_TOKENS = 4
EPOCHS         = 3
SEED           = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

# 이미지 전처리

InternVL2는 AutoProcessor 대신 직접 이미지를 448×448 타일로 분할합니다.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size=IMG_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB")),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_best_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=MAX_TILES, image_size=IMG_SIZE, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = sorted(
        {(i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1]
    )
    target_aspect_ratio = find_best_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width  = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized = image.resize((target_width, target_height))
    tiles = []
    for i in range(blocks):
        row = i // target_aspect_ratio[0]
        col = i % target_aspect_ratio[0]
        box = (col * image_size, row * image_size, (col + 1) * image_size, (row + 1) * image_size)
        tiles.append(resized.crop(box))
    if use_thumbnail and len(tiles) != 1:
        thumbnail = image.resize((image_size, image_size))
        tiles.append(thumbnail)
    return tiles

def load_image(image_path_or_pil, max_num=MAX_TILES):
    if isinstance(image_path_or_pil, str):
        image = Image.open(image_path_or_pil).convert("RGB")
    else:
        image = image_path_or_pil.convert("RGB")
    transform = build_transform()
    tiles = dynamic_preprocess(image, max_num=max_num, use_thumbnail=True)
    pixel_values = torch.stack([transform(tile) for tile in tiles])  # [tiles, C, H, W]
    return pixel_values

print("이미지 전처리 함수 정의 완료")

# 모델, Tokenizer

26B 모델 다운로드: 약 50GB, 20~30분 소요됩니다.

In [ ]:
# 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
tokenizer.padding_side = "right"

# 모델 로드
base_model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

IMG_CONTEXT_TOKEN = "<IMG_CONTEXT>"
IMG_START_TOKEN   = "<img>"
IMG_END_TOKEN     = "</img>"
img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
base_model.img_context_token_id = img_context_token_id

NUM_IMAGE_TOKEN = base_model.num_image_token
print("NUM_IMAGE_TOKEN:", NUM_IMAGE_TOKEN)
print(f"IMG_CONTEXT_TOKEN id: {img_context_token_id}")

# 핵심: outer model이 아니라 inner language_model만 준비
base_model.language_model = prepare_model_for_kbit_training(base_model.language_model)
base_model.language_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "attention.wqkv",
        "attention.wo",
        "feed_forward.w1",
        "feed_forward.w2",
        "feed_forward.w3",
    ],
    task_type="CAUSAL_LM",
)

base_model.language_model = get_peft_model(base_model.language_model, lora_config)
base_model.language_model.enable_input_require_grads()
base_model.language_model.print_trainable_parameters()

model = base_model

# 프롬프트 템플릿

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# [수정 1] <image> 태그 → <img><IMG_CONTEXT>×256×tiles</img> 형식으로 수정
def build_internvl_prompt(user_text, num_tiles):
    image_tokens = IMG_START_TOKEN + IMG_CONTEXT_TOKEN * NUM_IMAGE_TOKEN * num_tiles + IMG_END_TOKEN
    return (
        f"<|im_start|>system\n{SYSTEM_INSTRUCT}<|im_end|>\n"
        f"<|im_start|>user\n{image_tokens}\n{user_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# Custom Dataset, Collator

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, tokenizer, train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        pixel_values = load_image(row["path"])  # [tiles, C, H, W]
        num_tiles = pixel_values.shape[0]

        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        )
        prompt = build_internvl_prompt(user_text, num_tiles)

        if self.train:
            gold = str(row["answer"]).strip().lower()
            full_text = prompt + gold + tokenizer.eos_token
        else:
            gold = None
            full_text = prompt

        return {
            "pixel_values": pixel_values,
            "text": full_text,
            "prompt_text": prompt,
            "answer_text": gold,
        }


from dataclasses import dataclass
from typing import Any
import torch

@dataclass
class DataCollator:
    tokenizer: Any
    train: bool = True

    def __call__(self, batch):
        texts = [s["text"] for s in batch]
        prompt_texts = [s["prompt_text"] for s in batch]
        pixel_values_list = [s["pixel_values"] for s in batch]

        enc = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=2048,
            return_tensors="pt",
        )

        prompt_enc = self.tokenizer(
            prompt_texts,
            padding=True,
            truncation=True,
            max_length=2048,
            return_tensors="pt",
        )

        # batch_size=1 기준
        enc["pixel_values"] = torch.cat(pixel_values_list, dim=0)  # [tiles, C, H, W]

        enc["image_flags"] = torch.ones(
            enc["pixel_values"].shape[0],
            1,
            dtype=torch.long
        )

        if self.train:
            labels = enc["input_ids"].clone()

            # padding은 loss 제외
            labels[enc["attention_mask"] == 0] = -100

            # prompt 전체는 loss 제외, 정답+eos만 학습
            for i in range(len(batch)):
                prompt_len = int(prompt_enc["attention_mask"][i].sum().item())
                labels[i, :prompt_len] = -100

            enc["labels"] = labels

        return enc

# DataLoader

In [ ]:
split = int(len(train_df) * 0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

train_ds = VQAMCDataset(train_subset, tokenizer, train=True)
valid_ds = VQAMCDataset(valid_subset, tokenizer, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(tokenizer, True),
    num_workers=0
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(tokenizer, True),
    num_workers=0
)

# Fine-tuning

In [ ]:
from tqdm.auto import tqdm
import math
import torch

EPOCHS = 2

# gradient checkpointing과 충돌 방지
if hasattr(model, "config"):
    model.config.use_cache = False
if hasattr(model, "language_model") and hasattr(model.language_model, "config"):
    model.language_model.config.use_cache = False

model_device = next(model.parameters()).device
GRAD_ACCUM = 8

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(num_training_steps * 0.03),
    num_training_steps
)

global_step = 0
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    optimizer.zero_grad(set_to_none=True)

    nan_found = False

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(model_device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"NaN/Inf loss detected at epoch={epoch+1}, step={step}")
            nan_found = True
            break

        loss.backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    # 마지막 accumulation 잔여 step 처리
    if (not nan_found) and (len(train_loader) % GRAD_ACCUM != 0):
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    if nan_found:
        print(f"[Epoch {epoch+1}] stopped due to NaN/Inf loss")
        break

    model.eval()
    val_loss, val_steps = 0.0, 0
    val_nan_found = False

    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(model_device) for k, v in vb.items()}
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                vloss = model(**vb).loss

            if torch.isnan(vloss) or torch.isinf(vloss):
                print(f"NaN/Inf valid loss detected at epoch={epoch+1}")
                val_nan_found = True
                break

            val_loss += vloss.item()
            val_steps += 1

    if val_nan_found:
        print(f"[Epoch {epoch+1}] validation stopped due to NaN/Inf loss")
        break

    if val_steps > 0:
        print(f"[Epoch {epoch+1}] valid loss: {val_loss / val_steps:.4f}")

SAVE_DIR = "/content/internvl2_26b_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

In [ ]:
!zip -r /content/internvl2_26b_lora.zip /content/internvl2_26b_lora

In [ ]:
from google.colab import files
files.download("/content/internvl2_26b_lora.zip")

# Inference

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"

    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    for line in reversed(lines):
        tokens = line.replace(".", " ").replace(")", " ").split()
        for tok in tokens:
            if tok in ["a", "b", "c", "d"]:
                return tok

    return "a"


torch.cuda.empty_cache()
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]

    # 이미지 로드: [tiles, C, H, W]
    pixel_values = load_image(row["path"]).to(model_device, dtype=torch.bfloat16)
    num_tiles = pixel_values.shape[0]

    # 프롬프트 생성
    user_text = build_mc_prompt(
        row["question"], row["a"], row["b"], row["c"], row["d"]
    )
    prompt = build_internvl_prompt(user_text, num_tiles)

    # 텍스트 입력
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    # 이미지 입력
    inputs["pixel_values"] = pixel_values

    # InternVL2용 image_flags 추가
    inputs["image_flags"] = torch.ones(
        pixel_values.shape[0],
        1,
        dtype=torch.long,
        device=model_device
    )

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 프롬프트 부분 제거 후 생성 결과만 디코딩
    generated = out_ids[:, inputs["input_ids"].shape[1]:]
    output_text = tokenizer.decode(generated[0], skip_special_tokens=True)

    preds.append(extract_choice(output_text))

    del inputs, out_ids, pixel_values
    torch.cuda.empty_cache()

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds
})
submission.to_csv("/content/submission.csv", index=False)

print("Saved /content/submission.csv")
print(submission.head())
print("Last output_text:", output_text)

In [ ]:
# 마지막 추론 결과 확인
print(output_text)